# Functions

### Parse Frame And Sprites Functions

In [1]:
# Returns Parser With Frame and Spirte Objects



import struct
from typing import List, Optional, Tuple

class Sprite:
    """Class representing a single sprite in a Sega Genesis file."""
    
    # Sprite Size Lookup Table
    SIZE_TABLE = {
        0x0: {"tiles": 1, "dimensions": (1, 1), "value": 0x01},
        0x1: {"tiles": 2, "dimensions": (1, 2), "value": 0x02},
        0x2: {"tiles": 3, "dimensions": (1, 3), "value": 0x03},
        0x3: {"tiles": 4, "dimensions": (1, 4), "value": 0x04},
        0x4: {"tiles": 2, "dimensions": (2, 1), "value": 0x02},
        0x5: {"tiles": 4, "dimensions": (2, 2), "value": 0x04},
        0x6: {"tiles": 6, "dimensions": (2, 3), "value": 0x06},
        0x7: {"tiles": 8, "dimensions": (2, 4), "value": 0x08},
        0x8: {"tiles": 3, "dimensions": (3, 1), "value": 0x03},
        0x9: {"tiles": 6, "dimensions": (3, 2), "value": 0x06},
        0xA: {"tiles": 9, "dimensions": (3, 3), "value": 0x09},
        0xB: {"tiles": 12, "dimensions": (3, 4), "value": 0x0C},
        0xC: {"tiles": 4, "dimensions": (4, 1), "value": 0x04},
        0xD: {"tiles": 8, "dimensions": (4, 2), "value": 0x08},
        0xE: {"tiles": 12, "dimensions": (4, 3), "value": 0x0C},
        0xF: {"tiles": 16, "dimensions": (4, 4), "value": 0x10}
    }
    
    def __init__(self, data: bytes):
        """Initialize sprite with its 8-byte data."""
        if len(data) != 8:
            raise ValueError(f"Sprite data must be 8 bytes, got {len(data)} bytes")
        self.data = data
        
        # Parse the sprite fields according to the specification
        self.x_pos = struct.unpack('>h', data[0:2])[0]  # Signed 16-bit big-endian
        self.y_pos = struct.unpack('>h', data[2:4])[0]  # Signed 16-bit big-endian
        self.tile_offset = struct.unpack('>H', data[4:6])[0]  # Unsigned 16-bit
        self.palette_id = data[6]  # 1 byte
        self.sizetab = data[7]  # 1 byte
        
        # Get size information from the lookup table
        sizetab_index = self.sizetab & 0x0F  # Use only the lower 4 bits for lookup
        size_info = self.SIZE_TABLE.get(sizetab_index, {"tiles": 0, "dimensions": (0, 0), "value": 0})
        self.tile_count = size_info["tiles"]
        self.width_tiles, self.height_tiles = size_info["dimensions"]
        self.size_value = size_info["value"]
    
    def __str__(self) -> str:
        """Return a string representation of the sprite data."""
        return (f"X: {self.x_pos}, Y: {self.y_pos}, Tile: 0x{self.tile_offset:04X}, "
                f"Palette: 0x{self.palette_id:02X}, Size: {self.width_tiles}x{self.height_tiles} ({self.tile_count} tiles)")
    
    def __repr__(self) -> str:
        """Return a string representation for debugging."""
        return f"Sprite(x={self.x_pos}, y={self.y_pos}, tile=0x{self.tile_offset:04X}, size={self.width_tiles}x{self.height_tiles})"
    
    @property
    def hex_data(self) -> str:
        """Return the sprite data as a hex string."""
        return ' '.join([f"{b:02X}" for b in self.data])
    
    @property
    def detailed_info(self) -> str:
        """Return detailed information about the sprite."""
        return (
            f"Raw Data: {self.hex_data}\n"
            f"X Position: {self.x_pos} (0x{self.x_pos & 0xFFFF:04X})\n"
            f"Y Position: {self.y_pos} (0x{self.y_pos & 0xFFFF:04X})\n"
            f"Tile Offset: 0x{self.tile_offset:04X}\n"
            f"Palette ID: 0x{self.palette_id:02X}\n"
            f"Sizetab: 0x{self.sizetab:02X}\n"
            f"Sizetab Index: 0x{self.sizetab & 0x0F:X}\n"
            f"Size Value: 0x{self.size_value:02X}\n"
            f"Tile Count: {self.tile_count}\n"
            f"Dimensions: {self.width_tiles}x{self.height_tiles} tiles"
        )
    
    @property
    def dimensions(self) -> Tuple[int, int]:
        """Return the sprite dimensions in tiles as (width, height)."""
        return (self.width_tiles, self.height_tiles)


class Frame:
    """Class representing a frame in a Sega Genesis file."""
    
    def __init__(self, index: int, offset: int, file_offset: int, data: bytes):
        """Initialize a frame with its metadata and sprite data."""
        self.index = index  # Frame number
        self.offset = offset  # Original offset from the file
        self.file_offset = file_offset  # Offset relative to first frame
        self.data = data  # Raw data for this frame
        self.sprites: List[Sprite] = []
        
        # Initialize hotspot coordinates (will be set later)
        self.hot_x: Optional[int] = None
        self.hot_y: Optional[int] = None
        
        # Parse the sprite data into individual sprites (8 bytes each)
        for i in range(0, len(data), 8):
            if i + 8 <= len(data):
                sprite_data = data[i:i+8]
                self.sprites.append(Sprite(sprite_data))
    
    def __str__(self) -> str:
        """Return a string representation of the frame."""
        hot_info = f", hotspot=({self.hot_x}, {self.hot_y})" if self.hot_x is not None else ""
        return f"Frame {self.index}: offset=0x{self.offset:04X}, file_offset=0x{self.file_offset:04X}, sprites={len(self.sprites)}{hot_info}"
    
    def __repr__(self) -> str:
        """Return a string representation for debugging."""
        hot_info = f", hot=({self.hot_x}, {self.hot_y})" if self.hot_x is not None else ""
        return f"Frame({self.index}, 0x{self.offset:04X}, {len(self.sprites)} sprites{hot_info})"
    
    @property
    def sprite_count(self) -> int:
        """Return the number of sprites in this frame."""
        return len(self.sprites)
    
    @property
    def total_tiles(self) -> int:
        """Return the total number of tiles used by all sprites in this frame."""
        return sum(sprite.tile_count for sprite in self.sprites)
    
    def display_sprites(self, detailed: bool = False) -> None:
        """Display all sprites in this frame.
        
        Args:
            detailed: If True, show detailed information for each sprite.
        """
        print(f"Frame {self.index} (Offset: 0x{self.offset:04X}, FileOffset: 0x{self.file_offset:04X})")
        
        if self.hot_x is not None and self.hot_y is not None:
            print(f"Hotspot: X={self.hot_x}, Y={self.hot_y}")
        
        print(f"Contains {self.sprite_count} sprites with {self.total_tiles} total tiles:")
        
        if self.sprite_count == 0:
            print("  No sprites in this frame")
            return
        
        for i, sprite in enumerate(self.sprites):
            if detailed:
                print(f"  Sprite {i}:")
                for line in sprite.detailed_info.split('\n'):
                    print(f"    {line}")
                print()
            else:
                print(f"  Sprite {i:3d}: {sprite}")
        print()


class GenesisFileLoader:
    """Class for loading and parsing Sega Genesis sprite data files."""
    
    def __init__(self, offsets_file: str, sprite_data_file: str, hotlist_file: Optional[str] = None, sprite_tile_file: Optional[str] = None):
        """Initialize with paths to the offset file, sprite data file, and optional hotlist file."""
        self.offsets_file = offsets_file
        self.sprite_data_file = sprite_data_file
        self.hotlist_file = hotlist_file
        self.sprite_tile_file = sprite_tile_file
        self.frames: List[Frame] = []
        self.frame_offsets: List[int] = []
        self.first_offset: int = 0
    
    def load_files(self) -> None:
        """Load and parse all files."""
        self._load_offsets()
        self._load_sprite_data()
        if self.hotlist_file:
            self._load_hotlist()
    
    def _load_offsets(self) -> None:
        """Load frame offsets from the offsets file."""
        with open(self.offsets_file, 'rb') as f:
            offsets_data = f.read()
        
        # Skip the first 2 bytes (signature)
        offset = 2
        
        # Read each frame's offset (2 bytes each, big-endian)
        num_frames = (len(offsets_data) - 2) // 2
        
        for i in range(num_frames):
            frame_offset_bytes = offsets_data[offset:offset+2]
            frame_offset = struct.unpack('>H', frame_offset_bytes)[0]
            self.frame_offsets.append(frame_offset)
            offset += 2
        
        # Store the first offset for relative calculations
        if self.frame_offsets:
            self.first_offset = self.frame_offsets[0]
        
        print(f"Loaded {num_frames} frame offsets from {self.offsets_file}")
    
    def _load_sprite_data(self) -> None:
        """Load sprite data and create Frame objects."""
        if not self.frame_offsets:
            print("No frame offsets loaded. Call load_offsets() first.")
            return
        
        with open(self.sprite_data_file, 'rb') as f:
            sprite_data = f.read()
        
        num_frames = len(self.frame_offsets)
        
        for i in range(num_frames):
            offset = self.frame_offsets[i]
            file_offset = offset - self.first_offset
            
            # Calculate the length of data for this frame
            if i < num_frames - 1:
                next_offset = self.frame_offsets[i+1]
                data_length = next_offset - offset
            else:
                # For the last frame, assume it extends to the end of the file
                data_length = len(sprite_data) - file_offset
            
            # Extract this frame's data
            frame_data = sprite_data[file_offset:file_offset + data_length]
            
            # Create a Frame object
            frame = Frame(i, offset, file_offset, frame_data)
            self.frames.append(frame)
        
        print(f"Loaded {len(self.frames)} frames from {self.sprite_data_file}")
    
    def _load_hotlist(self) -> None:
        """Load hotspot data from the hotlist file and populate frame hotspot coordinates."""
        if not self.frames:
            print("No frames loaded. Call load_sprite_data() first.")
            return
        
        try:
            with open(self.hotlist_file, 'rb') as f:
                hotlist_data = f.read()
            
            # Skip the first 2 bytes (0x08 0x05 which are skipped according to spec)
            offset = 2
            
            # Number of frames to process (limited by either hotlist data or loaded frames)
            max_frames = min(len(self.frames), (len(hotlist_data) - 2) // 2)
            
            for i in range(max_frames):
                # Each frame has 2 bytes: Hot X and Hot Y (signed bytes)
                if offset + 1 < len(hotlist_data):
                    # Convert to signed byte values (-128 to 127)
                    hot_x = hotlist_data[offset]
                    if hot_x > 127:  # Convert to signed byte
                        hot_x -= 256
                    
                    hot_y = hotlist_data[offset + 1]
                    if hot_y > 127:  # Convert to signed byte
                        hot_y -= 256
                    
                    # Assign hotspot coordinates to the corresponding frame
                    self.frames[i].hot_x = hot_x
                    self.frames[i].hot_y = hot_y
                    
                    offset += 2
            
            print(f"Loaded hotspot data for {max_frames} frames from {self.hotlist_file}")
        
        except Exception as e:
            print(f"Error loading hotlist file: {e}")
    
    def display_summary(self) -> None:
        """Display a summary of the loaded data."""
        if not self.frames:
            print("No frames loaded. Call load_files() first.")
            return
        
        print(f"\nTotal number of frames: {len(self.frames)}")
        print(f"First frame offset (base): 0x{self.first_offset:04X} ({self.first_offset})")
        print("\nFrame Summary:")
        print("Frame | Offset (hex) | FileOffset (hex) | # of Sprites | Total Tiles | Hotspot")
        print("-" * 90)
        
        for frame in self.frames:
            hotspot_info = f"({frame.hot_x}, {frame.hot_y})" if frame.hot_x is not None else "None"
            print(f"{frame.index:5d} | 0x{frame.offset:04X}       | "
                  f"0x{frame.file_offset:04X}           | {frame.sprite_count:10d} | "
                  f"{frame.total_tiles:10d} | {hotspot_info}")
    
    def display_all_frames_and_sprites(self, detailed: bool = False) -> None:
        """Display detailed information for all frames and their sprites.
        
        Args:
            detailed: If True, show detailed information for each sprite.
        """
        if not self.frames:
            print("No frames loaded. Call load_files() first.")
            return
        
        print(f"\n{'='*80}")
        print(f"DETAILED VIEW OF ALL FRAMES AND SPRITES")
        print(f"{'='*80}")
        print(f"Total Frames: {len(self.frames)}")
        print(f"First Frame Offset (Base): 0x{self.first_offset:04X}")
        print(f"{'='*80}\n")
        
        for frame in self.frames:
            print(f"FRAME {frame.index}")
            print(f"{'-'*60}")
            print(f"Offset: 0x{frame.offset:04X} ({frame.offset})")
            print(f"FileOffset: 0x{frame.file_offset:04X} ({frame.file_offset})")
            
            if frame.hot_x is not None and frame.hot_y is not None:
                print(f"Hotspot: X={frame.hot_x}, Y={frame.hot_y}")
            
            print(f"Number of Sprites: {frame.sprite_count}")
            print(f"Total Tiles: {frame.total_tiles}")
            
            if frame.sprite_count > 0:
                print(f"\nSprites:")
                frame.display_sprites(detailed=detailed)
            else:
                print(f"\nNo sprites in this frame")
            
            print(f"\n{'='*80}\n")
    
    def display_frame_range(self, start_frame: int, end_frame: int, detailed: bool = False) -> None:
        """Display detailed information for a range of frames.
        
        Args:
            start_frame: The starting frame index.
            end_frame: The ending frame index (inclusive).
            detailed: If True, show detailed information for each sprite.
        """
        if not self.frames:
            print("No frames loaded. Call load_files() first.")
            return
        
        # Validate the range
        start_frame = max(0, min(start_frame, len(self.frames) - 1))
        end_frame = max(start_frame, min(end_frame, len(self.frames) - 1))
        
        print(f"\n{'='*80}")
        print(f"DETAILED VIEW OF FRAMES {start_frame} to {end_frame}")
        print(f"{'='*80}")
        
        for i in range(start_frame, end_frame + 1):
            frame = self.frames[i]
            print(f"FRAME {frame.index}")
            print(f"{'-'*60}")
            print(f"Offset: 0x{frame.offset:04X} ({frame.offset})")
            print(f"FileOffset: 0x{frame.file_offset:04X} ({frame.file_offset})")
            
            if frame.hot_x is not None and frame.hot_y is not None:
                print(f"Hotspot: X={frame.hot_x}, Y={frame.hot_y}")
            
            print(f"Number of Sprites: {frame.sprite_count}")
            print(f"Total Tiles: {frame.total_tiles}")
            
            if frame.sprite_count > 0:
                print(f"\nSprites:")
                frame.display_sprites(detailed=detailed)
            else:
                print(f"\nNo sprites in this frame")
            
            print(f"\n{'='*80}\n")
    
    def get_frame(self, index: int) -> Optional[Frame]:
        """Get a specific frame by index."""
        if 0 <= index < len(self.frames):
            return self.frames[index]
        return None
    
    def get_frame_stats(self) -> dict:
        """Get statistics about all frames."""
        if not self.frames:
            return {"error": "No frames loaded"}
        
        total_sprites = sum(frame.sprite_count for frame in self.frames)
        total_tiles = sum(frame.total_tiles for frame in self.frames)
        max_sprites_frame = max(self.frames, key=lambda f: f.sprite_count)
        max_tiles_frame = max(self.frames, key=lambda f: f.total_tiles)
        
        return {
            "total_frames": len(self.frames),
            "total_sprites": total_sprites,
            "total_tiles": total_tiles,
            "average_sprites_per_frame": total_sprites / len(self.frames),
            "average_tiles_per_frame": total_tiles / len(self.frames),
            "max_sprites": {
                "count": max_sprites_frame.sprite_count,
                "frame": max_sprites_frame.index
            },
            "max_tiles": {
                "count": max_tiles_frame.total_tiles,
                "frame": max_tiles_frame.index
            },
            "frames_with_hotspots": sum(1 for frame in self.frames if frame.hot_x is not None)
        }

### Create PNG Functions

### Call Functions

In [2]:
# Usage example
loader = GenesisFileLoader(
    offsets_file='94FrameSpriteDataOffsets.bin',
    sprite_data_file='94SpriteDataBytes.bin',
    hotlist_file='94FrameHotList.bin',
    sprite_tile_file='94SpriteTileData.bin'
)
loader.load_files()

# Display summary
#loader.display_summary()

# Display detailed information for all frames and sprites
loader.display_all_frames_and_sprites(detailed=True)


Loaded 846 frame offsets from 94FrameSpriteDataOffsets.bin
Loaded 846 frames from 94SpriteDataBytes.bin
Loaded hotspot data for 837 frames from 94FrameHotList.bin

DETAILED VIEW OF ALL FRAMES AND SPRITES
Total Frames: 846
First Frame Offset (Base): 0x069E

FRAME 0
------------------------------------------------------------
Offset: 0x069E (1694)
FileOffset: 0x0000 (0)
Hotspot: X=-4, Y=-20
Number of Sprites: 1
Total Tiles: 8

Sprites:
Frame 0 (Offset: 0x069E, FileOffset: 0x0000)
Hotspot: X=-4, Y=-20
Contains 1 sprites with 8 total tiles:
  Sprite 0:
    Raw Data: FF F9 FF E7 00 00 40 07
    X Position: -7 (0xFFF9)
    Y Position: -25 (0xFFE7)
    Tile Offset: 0x0000
    Palette ID: 0x40
    Sizetab: 0x07
    Sizetab Index: 0x7
    Size Value: 0x08
    Tile Count: 8
    Dimensions: 2x4 tiles




FRAME 1
------------------------------------------------------------
Offset: 0x06A6 (1702)
FileOffset: 0x0008 (8)
Hotspot: X=-1, Y=-24
Number of Sprites: 2
Total Tiles: 9

Sprites:
Frame 1 (Offse